# 01. Reviewer demands - analyses over the v7 extraction

## 01. Introduction

This notebook reads the analytic base of the systematic review (JHPN, major revision) and produces the numbers, tables and figures required by the quantitative demands of the reports (Editor, Reviewers 1, 2 and 3). Input: `methods/extracao_artigo1_v7.xlsx`, sheet `Per Combination` (241 rows of study x outcome x algorithm, 38 studies, CHARMS fields 1-10, PROBAST+AI and "3 Most Important Features"). The current manuscript uses the old base (43 studies, 47 combinations); every number has to be recomputed over the base of 38 studies.

Products, written to `jhpn_revisions/analysis/resultados/`:

1. **Units of analysis:** `unidades_analise.csv` and `auditoria_desfechos_R2_5.csv` (outcomes extracted outside the declared scope).
2. **Corpus:** `corpus_descritivo.csv` and the year x outcome figure (new Figure 2A).
3. **Algorithms:** `algoritmos.csv` (families, random forest, deep learning).
4. **Performance:** `table2_nova.csv`, `desempenho_estratificado.csv` and a boxplot of AUC by outcome.
5. **Class imbalance:** `desbalanceamento.csv` and `heterogeneidade.csv` (the basis for not running a meta-analysis).
6. **Model purpose:** `classificacao_finalidade_TEMPLATE.csv` (case by case, for IVS to check).
7. **Leakage:** `leakage_recontado.csv` and the figure of most important predictors (new Figure 4).
8. **PROBAST+AI:** `probast_dominios.csv`.
9. **Synthesis:** `numeros_manuscrito.csv` (old value x new value, per statement in the text).

Rules: v7 is read-only and is never written to. NR never becomes a number (it becomes NaN and leaves the denominator, which is always reported). The denominator of every number is explicit: study (n=38), study x outcome combination, or study x outcome x algorithm row (n=241). Eligible outcomes of the article: stunting, obesity, overweight, thinness; wasting and underweight were excluded by a protocol decision (T1.5) and appear only in the audit. The checking template is never rewritten once it already carries marks.

## 02. Libraries and Mount

Colab-or-local mount and a single cell that installs and imports every library.

In [ ]:
# -- 02.1 Mount (Colab ou local) --
import os
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/USP/Doutorado/Coorte de pelotas/01_artigos/artigo_1'
    AMBIENTE = 'colab'
except ModuleNotFoundError:
    BASE = '../..'
    AMBIENTE = 'local'
V7 = f'{BASE}/methods/extracao_artigo1_v7.xlsx'
OUT_DIR = f'{BASE}/jhpn_revisions/analysis/resultados'
TEMPLATE = f'{OUT_DIR}/classificacao_finalidade_TEMPLATE.csv'
AUDITORIA = f'{OUT_DIR}/auditoria_v7_por_estudo.csv'
DIVERGENCIAS = f'{OUT_DIR}/auditoria_v7_divergencias.csv'
BASE_DADOS = f'{BASE}/base_dados'
FIGURAS = f'{BASE}/jhpn_revisions/figures'
DATA = '20260826'
os.makedirs(OUT_DIR, exist_ok=True)
print(AMBIENTE, '| BASE =', BASE)

In [ ]:
# -- 02.2 libraries: install everything at once and import --
%pip install -q openpyxl plotly "kaleido==0.2.1"  # kaleido 0.2.1 embeds Chromium; kaleido>=1 needs to download Chrome and fails on Colab without open network access. No -U: upgrading matplotlib/pandas breaks backend_pdf
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 220)

def grava(tab, nome):
    tab.to_csv(f'{OUT_DIR}/{nome}', sep=';', encoding='utf-8-sig', index=False)
    print(f'written: resultados/{nome} ({tab.shape[0]} rows x {tab.shape[1]} columns)')

# -- single figure style (palette validated for colour blindness; labels in English) --
CORES = {'stunting': '#2A6FB0', 'obesity': '#C2453A',
         'overweight': '#E08A2E', 'overweight/obesity (combined)': '#7B5EA7'}
ROTULO_EN = {'stunting': 'Stunting', 'obesity': 'Obesity',
             'overweight': 'Overweight', 'overweight/obesity (combined)': 'Overweight/obesity'}
TINTA, TINTA2, GRADE_COR = '#222222', '#555555', '#DDDDDD'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.titlesize': 10, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelsize': 9, 'axes.labelcolor': TINTA, 'axes.edgecolor': TINTA2,
    'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': TINTA2, 'ytick.color': TINTA2, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5,
    'grid.color': GRADE_COR, 'grid.linewidth': 0.7, 'text.color': TINTA,
})


# label placed at the end of the bar, instead of a number inside each stacked segment
def rotula_barras(ax, retangulos, valores, dx=0, dy=0, cor=TINTA, tam=8, horizontal=False):
    for r, v in zip(retangulos, valores):
        if not v:
            continue
        if horizontal:
            ax.text(r.get_width() + dx, r.get_y() + r.get_height() / 2, f'{v:g}',
                    va='center', ha='left', fontsize=tam, color=cor)
        else:
            ax.text(r.get_x() + r.get_width() / 2, r.get_height() + dy, f'{v:g}',
                    ha='center', va='bottom', fontsize=tam, color=cor)


def fig_salva(fig, nome):
    fig.savefig(f'{OUT_DIR}/{nome}_{DATA}.png', dpi=200, bbox_inches='tight')
    try:
        fig.savefig(f'{OUT_DIR}/{nome}_{DATA}.pdf', bbox_inches='tight')
        print(f'figure: resultados/{nome}_{DATA}.png/.pdf')
    except Exception as e:  # backend_pdf quebra se a matplotlib foi trocada na sessao
        print(f'figure: resultados/{nome}_{DATA}.png (PDF not generated: {type(e).__name__})')

## 03. Opening the extraction file

Row 1 of the spreadsheet is the band of CHARMS groups; the real header is row 2 (`header=1`). The standardised columns become numeric (NR turns into NaN). Three views: `df` (241 rows of study x outcome x algorithm), `df_comb` (one row per study x outcome combination, chosen by the best-model rule) and `df_est` (38 studies).

In [ ]:
# -- 03.1 reading and types --
df = pd.read_excel(V7, sheet_name='Per Combination', header=1)
# AA_10 (Kar, 2021) modelled only a composite undernutrition outcome, with no isolable
# anthropometric dimension. It is ineligible, and leaves every count.
EXCLUIDOS = ['AA_10']
df = df[~df['ID'].isin(EXCLUIDOS)].copy()
METRICAS = ['AUC', 'Accuracy', 'Precision', 'Recall (Sensitivity)', 'Specificity', 'F1-Score']
NUMERICAS = METRICAS + ['Sex (%F)', 'N (sample)', 'N (analysis)', 'N Events', 'EPV (calculated)', 'Events Denominator']
for c in NUMERICAS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
assert df.shape[0] == 237, f'expected 237 rows, got {df.shape[0]}'
n_estudos = df['ID'].nunique()
assert n_estudos == 37, f'expected 37 studies, got {n_estudos}'
fora = pd.concat([df[m][(df[m] < 0) | (df[m] > 1)] for m in METRICAS])
assert fora.empty, 'metrica padronizada fora de [0,1]'
print(f'v7: {df.shape[0]} rows x {df.shape[1]} columns | {df["ID"].nunique()} studies | metrics in [0,1] ok')

In [ ]:
# -- 03.2 outcome family and derived prevalence --
def familia_desfecho(s):
    t = str(s).lower()
    if 'overweight/obesity' in t:
        return 'overweight/obesity (combined)'
    if 'obes' in t:
        return 'obesity'
    if 'overweight' in t:
        return 'overweight'
    if 'stunt' in t or 'height-for-age' in t:
        return 'stunting'
    if 'wasting' in t:
        return 'wasting'
    if 'underweight' in t:
        return 'underweight'
    if 'thin' in t:
        return 'thinness'
    if 'malnutrition' in t:
        return 'malnutrition (composto)'
    return 'other'

df['outcome_family'] = df['Outcome'].map(familia_desfecho)
ELEGIVEIS = ['stunting', 'obesity', 'overweight', 'overweight/obesity (combined)']  # thinness left the scope (26 Aug); the combined label is overweight/obesity and it stays
df['eligible'] = df['outcome_family'].isin(ELEGIVEIS)
denom = df['Events Denominator'].fillna(df['N (analysis)']).fillna(df['N (sample)'])
df['outcome_prevalence'] = df['N Events'] / denom
assert (df.loc[df['outcome_prevalence'].notna(), 'outcome_prevalence'].between(0, 1)).all(), 'prevalence outside [0,1]'
print(df['outcome_family'].value_counts().to_string())
print(f"eligible rows: {int(df['eligible'].sum())}/241 | prevalence computable in {int(df['outcome_prevalence'].notna().sum())} rows")

In [ ]:
# -- 03.3 views: combination (best model) and study --
linhas, via_yes = [], 0
for (sid, fam), g in df.groupby(['ID', 'outcome_family']):
    g = g.copy()
    g['_yes'] = g['Best Model?'].astype(str).str.strip().str.lower().eq('yes')
    g = g.sort_values(['_yes', 'AUC', 'Accuracy', 'F1-Score'],
                      ascending=[False, False, False, False], na_position='last')
    via_yes += int(g['_yes'].iloc[0])
    linhas.append(g.iloc[0].drop('_yes'))
df_comb = pd.DataFrame(linhas).reset_index(drop=True)
df_comb_eleg = df_comb[df_comb['eligible']].copy()
df_est = df.drop_duplicates('ID').copy()
print(f'study x outcome combinations: {len(df_comb)} (eligible: {len(df_comb_eleg)})')
print(f'combinations resolved by the Best Model = Yes mark: {via_yes}; by highest AUC/Accuracy: {len(df_comb) - via_yes}')
print(f'studies: {len(df_est)}')

## 04. Units of analysis and the canonical N

The manuscript says "43 studies representing 47 study-outcome combinations". The current base has 38 studies. This section holds the new N per outcome and the audit of demand R2.5: Reviewer 2 pointed out an inconsistency about wasting and underweight (the objectives include them, later sections exclude them). The CHARMS extraction recorded EVERY outcome reported by the articles; those outside the declared scope go to `auditoria_desfechos_R2_5.csv`, for IVS to decide the wording of the text (the exclusion is declared in comments T1.5).

In [ ]:
# -- 04.1 combinations by outcome: new vs manuscript --
ANTIGO = {'stunting': 32, 'obesity': 9, 'overweight': 5, 'thinness': 1}  # current Table 1 (47 combinations)
novo = df_comb['outcome_family'].value_counts()
uni = pd.DataFrame({'outcome': novo.index,
                    'combinations_v7': novo.values,
                    'combinations_manuscript_47': [ANTIGO.get(f, 0) for f in novo.index],
                    'eligible_in_article': [f in ELEGIVEIS for f in novo.index]})
uni['studies_v7'] = [df_comb.loc[df_comb['outcome_family'] == f, 'ID'].nunique() for f in uni['outcome']]
uni = uni.sort_values('combinations_v7', ascending=False).reset_index(drop=True)
print(uni.to_string(index=False))
n_comb_eleg = len(df_comb_eleg)
n_est_eleg = df_comb_eleg['ID'].nunique()
print(f'\nproposed canonical N: {n_est_eleg} studies with an eligible outcome, {n_comb_eleg} eligible combinations')
print(f'(current manuscript: 43 studies, 47 combinations; PROBAST already uses 38)')
grava(uni, 'unidades_analise.csv')

In [ ]:
# -- 04.2 audit R2.5: outcomes extracted outside the declared scope --
aud = df_comb[~df_comb['eligible']][['ID', 'First Author', 'Year', 'outcome_family', 'Outcome']].copy()
aud = aud.sort_values(['outcome_family', 'ID']).reset_index(drop=True)
aud['suggested_action'] = 'outside the declared scope (T1.5): does not enter the numbers of the article; cite the exclusion in Methods'
print(aud[['ID', 'First Author', 'outcome_family']].to_string(index=False))
n_est_fora = aud['ID'].nunique()
so_fora = set(df_comb['ID']) - set(df_comb_eleg['ID'])
print(f'\n{len(aud)} combinations outside the scope, in {n_est_fora} studies')
print(f'studies WITHOUT any eligible outcome (do they enter the article?): {sorted(so_fora) if so_fora else "none"}')
print('note: thinness does not appear in v7; the single thinness study of the old base is not among the 38')
grava(aud, 'auditoria_desfechos_R2_5.csv')

## 05. Corpus

Descriptives of the studies and of the eligible combinations: countries (spellings normalised; the text says "15 countries" with the USA under 3 spellings), the concentration in Indonesia (R1.3 and R2.2 use different denominators: study and combination), year of publication, data source and sample size. The year x outcome figure replaces Figure 2A.

In [ ]:
# -- 05.1 normalised countries and Indonesia --
def pais_norm(s):
    t = str(s).strip()
    if t.upper().startswith('USA') or t == 'United States':
        return 'USA'
    if t.lower().startswith('asia'):
        return 'Asia (multinational dataset)'
    return t

df['country'] = df['Country/Region'].map(pais_norm)
df_comb_eleg['country'] = df_comb_eleg['Country/Region'].map(pais_norm)
df_est['country'] = df_est['Country/Region'].map(pais_norm)
paises = df_est['country'].value_counts()
print(paises.to_string())
n_paises = df_est.loc[df_est['country'] != 'Asia (multinational dataset)', 'country'].nunique()
print(f'\ndistinct countries (not counting the multinational dataset): {n_paises} (the manuscript says 15)')
ind_e = int((df_est['country'] == 'Indonesia').sum())
ind_c = int((df_comb_eleg['country'] == 'Indonesia').sum())
print(f'Indonesia: {ind_e}/{len(df_est)} studies ({100*ind_e/len(df_est):.0f}%) | '
      f'{ind_c}/{n_comb_eleg} eligible combinations ({100*ind_c/n_comb_eleg:.0f}%) | manuscript: 23/47 (49%)')

In [ ]:
# -- 05.2 year, data source and sample --
def fonte(s):
    t = str(s).lower()
    if 'dhs' in t or 'mics' in t or 'household survey' in t or 'health survey' in t or 'ensanut' in t or 'lsms' in t:
        return 'population survey (DHS/MICS/national)'
    if 'ehr' in t or 'electronic health record' in t:
        return 'electronic health record (EHR)'
    if 'kaggle' in t or 'open dataset' in t or 'open-source dataset' in t:
        return 'open dataset (Kaggle/not specified)'
    if 'school' in t:
        return 'school survey'
    if 'panel' in t or 'family life survey' in t:
        return 'longitudinal household panel'
    if 'health center' in t or 'health service' in t or 'growth monitoring' in t or 'posyandu' in t or 'puskesmas' in t or 'health office' in t or 'checkup' in t or 'nutrition survey' in t:
        return 'local health service (routine records)'
    return 'outra'

df_est['data_source'] = df_est['Source Type'].map(fonte)
print(df_est['data_source'].value_counts().to_string())
anos = df_est['Year'].value_counts().sort_index()
print('\nyear of publication (studies):', dict(anos))
pico = anos.idxmax()
print(f'peak: {int(anos.max())}/{len(df_est)} studies in {pico} ({100*anos.max()/len(df_est):.0f}%) | manuscript: 26/47 (55%) in 2024')
ns = df_est['N (sample)'].dropna()
print(f'\nsample size ({len(ns)}/38 studies with N): {int(ns.min())} to {int(ns.max())}, median {int(ns.median())} '
      f'| manuscript: 114 to 244,053, median 6,677')
corpus = df_est[['ID', 'First Author', 'Year', 'country', 'data_source', 'N (sample)', 'Age (months)', 'Setting']].copy()
grava(corpus.sort_values('ID'), 'corpus_overview.csv')

In [ ]:
# -- 05.3 Figure 2: (A) year x outcome, (B) world map --
tab_af = df_comb_eleg.pivot_table(index='Year', columns='outcome_family', values='ID', aggfunc='count').fillna(0)
tab_af = tab_af[[c for c in ELEGIVEIS if c in tab_af.columns]]
anos_rot = [str(int(a)) for a in tab_af.index]  # distinct name: 'anos' is the series from section 05.2

fig = plt.figure(figsize=(9, 8.2))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.25], hspace=0.28)
ax = fig.add_subplot(gs[0])
fundo = np.zeros(len(tab_af))
for c in tab_af.columns:
    ax.bar(anos_rot, tab_af[c], bottom=fundo, label=ROTULO_EN[c], color=CORES[c],
           width=0.62, edgecolor='white', linewidth=1.4)
    fundo += tab_af[c].values
for x, tot in zip(anos_rot, fundo):
    ax.text(x, tot + 0.4, f'{int(tot)}', ha='center', va='bottom', fontsize=8.5, color=TINTA)
ax.set_xlabel('Year of publication')
ax.set_ylabel('Study-outcome combinations')
ax.set_title('A. Temporal distribution', pad=24)
ax.set_ylim(0, fundo.max() * 1.18)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(ncols=4, loc='lower left', bbox_to_anchor=(0, 1.005), handlelength=1.1, handleheight=1.1,
          columnspacing=1.4, borderpad=0)
ax.set_axisbelow(True)
ax.yaxis.grid(True)

# panel B: map by country. If plotly/kaleido are unavailable, it falls back to ordered bars.
por_pais = (df_comb_eleg[df_comb_eleg['country'] != 'Asia (multinational dataset)']
            .groupby('country')['ID'].count().sort_values(ascending=False))
NOME_MAPA = {'USA': 'United States'}
bx = fig.add_subplot(gs[1])
bx.set_title('B. Geographic distribution', pad=6)
try:
    import plotly.express as px
    mapa = px.choropleth(
        pd.DataFrame({'country': [NOME_MAPA.get(p, p) for p in por_pais.index], 'n': por_pais.values}),
        locations='country', locationmode='country names', color='n',
        color_continuous_scale=[[0, '#DCE7F2'], [0.5, '#5D93C4'], [1, '#1F4E79']],
        range_color=(0, int(por_pais.max())))
    # country outline reinforced (31 Aug): a white border of 0.4 did not separate neighbouring countries
    mapa.update_traces(marker_line_color='#37474F', marker_line_width=0.9)
    mapa.update_geos(showframe=False, projection_type='robinson',
                     showland=True, landcolor='#F0F0F0', showlakes=False,
                     showcountries=True, countrycolor='#90A4AE', countrywidth=0.6,
                     showcoastlines=True, coastlinecolor='#37474F', coastlinewidth=0.9,
                     lataxis_range=[-58, 80], lonaxis_range=[-168, 182])
    mapa.update_layout(margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='white',
                       coloraxis_colorbar=dict(title='Combinations', thickness=26, len=0.72,
                                               tickfont=dict(size=26), title_font=dict(size=26),
                                               outlinewidth=0, x=0.99))
    mapa.write_image(f'{OUT_DIR}/_mapa_tmp.png', width=1700, height=800, scale=2)
    bx.imshow(plt.imread(f'{OUT_DIR}/_mapa_tmp.png'))
    bx.axis('off')
    os.remove(f'{OUT_DIR}/_mapa_tmp.png')
    resto = por_pais.iloc[3:]
    linha_rank = ' · '.join(f'{NOME_MAPA.get(k, k)} {v}' for k, v in por_pais.head(3).items())
    linha_rank += f' · {len(resto)} other countries {int(resto.sum())}'
    bx.text(0.5, 0.02, linha_rank, transform=bx.transAxes, ha='center', va='top',
            fontsize=8.5, color=TINTA2)
    modo_mapa = 'mapa'
except Exception as e:
    ordem = por_pais.sort_values()
    barras = bx.barh(ordem.index, ordem.values, color='#5D93C4', height=0.7)
    rotula_barras(bx, barras, ordem.values, dx=0.15, horizontal=True)
    bx.set_xlabel('Study-outcome combinations')
    bx.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    modo_mapa = f'barras ({type(e).__name__}: {e})'
    print(f'AVISO: painel B caiu no fallback de barras em vez do mapa. Motivo: {type(e).__name__}: {e}')
fig_salva(fig, 'fig2_distribuicao')
plt.show()
print(f'panel B: {modo_mapa} | countries: {len(por_pais)} | combinations: {int(por_pais.sum())}')

## 06. Algorithms

Demand R1.1: why simple ensembles dominate and deep learning appears so little. Frequency by algorithm family under three denominators (row, eligible combination by best model, study), plus reported tuning and imbalance handling.

In [ ]:
# -- 06.1 algorithm family --
def familia_algoritmo(s):
    t = str(s).lower()
    if 'stack' in t or 'voting' in t:
        return 'heterogeneous ensemble (stacking/voting)'
    if 'lstm' in t or 'deep learning' in t or 'dnn' in t or 'deep neural' in t:
        return 'deep learning'
    if 'boost' in t or 'xgb' in t or 'lgb' in t or 'catboost' in t:
        return 'boosting'
    if 'random forest' in t or 'bagging' in t:
        return 'random forest/bagging'
    if 'neural' in t or 'mlp' in t:
        return 'shallow neural network'
    if 'logistic' in t or 'lasso' in t or 'ridge' in t or 'elastic' in t or 'lda' in t or 'mars' in t:
        return 'regression/linear'
    if 'svm' in t or 'support vector' in t:
        return 'SVM'
    if 'nearest neighbor' in t or 'knn' in t or 'k-nn' in t:
        return 'KNN'
    if 'naive bayes' in t or 'gnb' in t or 'bnb' in t:
        return 'naive bayes'
    if 'tree' in t or 'cart' in t or 'ctree' in t:
        return 'decision tree'
    if 'gaussian process' in t:
        return 'gaussian process'
    return 'other'

df['algorithm_family'] = df['Algorithm'].map(familia_algoritmo)
df_comb_eleg['algorithm_family'] = df_comb_eleg['Algorithm'].map(familia_algoritmo)
sobra = df.loc[df['algorithm_family'] == 'other', 'Algorithm'].unique()
assert len(sobra) == 0, f'algorithms with no family: {sobra}'
alg = pd.DataFrame({'rows_in_extraction': df['algorithm_family'].value_counts(),
                    'combinations_as_best_model': df_comb_eleg['algorithm_family'].value_counts(),
                    'studies_using': df.groupby('algorithm_family')['ID'].nunique()}).fillna(0).astype(int)
alg = alg.sort_values('rows_in_extraction', ascending=False)
print(alg.to_string())
eh_rf = df['Algorithm'].str.lower().str.contains('random forest')
rf = int(eh_rf.sum())
rf_est = df.loc[eh_rf, 'ID'].nunique()
rf_melhor = int(df_comb_eleg['Algorithm'].str.lower().str.contains('random forest').sum())
dl_est = df.loc[df['algorithm_family'] == 'deep learning', 'ID'].unique()
print(f'\nRandom Forest: {rf}/241 rows ({100*rf/241:.0f}%); used in {rf_est}/38 studies '
      f'({100*rf_melhor/n_comb_eleg:.0f}%); best model in {rf_melhor}/{n_comb_eleg} eligible combinations '
      f'({100*rf_melhor/n_comb_eleg:.0f}%) | manuscript: 16/47 (34%), with no declared denominator')
print(f'deep learning: {len(dl_est)} studies ({list(dl_est)}) | manuscript: two studies')
grava(alg.reset_index().rename(columns={'index': 'familia_algoritmo'}), 'algorithm_families.csv')

In [ ]:
# -- 06.2 reported tuning and imbalance handling (by study) --
def reportado(s):
    t = str(s).strip().lower()
    return not (t in ('nr', 'nan', '') or t.startswith('nr ') or t.startswith('none') or t.startswith('no ') or t.startswith('not '))

tun = df_est['Hyperparameter Tuning'].map(reportado)
imb = df_est['Imbalance Handling'].map(reportado)
smote = df_est['Imbalance Handling'].astype(str).str.lower().str.contains('smote')
print(f'hyperparameter tuning reported: {int(tun.sum())}/38 studies ({100*tun.mean():.0f}%)')
print(f'imbalance handling reported: {int(imb.sum())}/38 ({100*imb.mean():.0f}%), of which SMOTE: {int(smote.sum())}')

## 07. Performance by outcome, stratified

Demand R2.8 (and comment 22 in the Doc): comparing median AUC across outcomes can mislead, because populations, prevalences, samples and validation schemes differ. The new Table 2 gives N, median and IQR by outcome; the strata by type of validation show how much performance depends on the design, and not only on the outcome. Unit: eligible combination, best model.

In [ ]:
# -- 07.1 Table 2 nova --
def r2(x):
    # a single rounding path in the whole notebook: round() and formatting diverge at
    # values exactly at the halfway point (the median AUC for obesity is 0.785 -> 0.78 vs 0.79)
    return float(f'{x:.2f}') if pd.notna(x) else np.nan

def resumo_metrica(g, col):
    v = g[col].dropna()
    if len(v) == 0:
        return pd.Series({f'{col}_N': 0, f'{col}_mediana': np.nan, f'{col}_IQR': ''})
    return pd.Series({f'{col}_N': len(v), f'{col}_mediana': r2(v.median()),
                      f'{col}_IQR': f'{v.quantile(.25):.2f}-{v.quantile(.75):.2f}'})

blocos = []
for fam, g in df_comb_eleg.groupby('outcome_family'):
    linha = pd.concat([resumo_metrica(g, m) for m in ['AUC', 'Accuracy', 'F1-Score']])
    linha['outcome'] = fam
    linha['combinations'] = len(g)
    blocos.append(linha)
t2 = pd.DataFrame(blocos).set_index('outcome').sort_values('combinations', ascending=False)
print(t2.to_string())
auc_rep = int(df_comb_eleg['AUC'].notna().sum())
print(f'\nAUC reported: {auc_rep}/{n_comb_eleg} combinations ({100*auc_rep/n_comb_eleg:.0f}%) | manuscript: 20/47 (43%)')
print('manuscript reference: stunting AUC 0.72 (0.63-0.87) n=8; obesity 0.81 (0.78-0.83) n=7; '
      'overweight 0.66 (0.62-0.73) n=4; accuracy stunting 0.88 (0.84-0.99) n=31')
grava(t2.reset_index(), 'table2_nova.csv')

In [ ]:
# -- 07.2 strata by type of validation --
def cat_validacao(s):
    t = str(s).lower()
    if 'external' in t:
        return 'external'
    if t.startswith('nr') or t in ('nan', ''):
        return 'NR/apparent'
    tem_kf, tem_ho = 'k-fold' in t or 'cv' in t, 'holdout' in t or 'split' in t
    so_tuning = 'tuning' in t or 'hyperparameter' in t or 'training' in t or 'up-sampling' in t
    if tem_kf and tem_ho and not so_tuning:
        return 'mixed/unclear'
    if tem_ho:
        return 'holdout'
    if tem_kf:
        return 'k-fold CV'
    return 'outra'

df_comb_eleg['validation'] = df_comb_eleg['Validation Type'].map(cat_validacao)
print(df_comb_eleg['validation'].value_counts().to_string())
ext = df_comb_eleg[df_comb_eleg['validation'] == 'external']
print(f'\nexternal validation: {ext["ID"].nunique()} study(ies) ({list(ext["ID"].unique())}) | manuscript: one study (2%)')
estr = (df_comb_eleg.groupby(['outcome_family', 'validation'])
        .agg(combinacoes=('ID', 'count'), AUC_mediana=('AUC', 'median'),
             Accuracy_mediana=('Accuracy', 'median'), N_mediano=('N (sample)', 'median'))
        .reset_index())
for c in ['AUC_median', 'Accuracy_median']:
    estr[c] = estr[c].map(r2)
print()
print(estr.to_string(index=False))
grava(estr, 'performance_by_validation.csv')

In [ ]:
# -- 07.3 Figure S1: AUC by outcome (points + box) --
fams = [f for f in ELEGIVEIS if df_comb_eleg.loc[df_comb_eleg['outcome_family'] == f, 'AUC'].notna().any()]
dados = [df_comb_eleg.loc[df_comb_eleg['outcome_family'] == f, 'AUC'].dropna() for f in fams]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
cx = ax.boxplot(dados, widths=0.45, showfliers=False, patch_artist=True,
                medianprops=dict(color=TINTA, linewidth=1.6),
                whiskerprops=dict(color=TINTA2, linewidth=0.9),
                capprops=dict(color=TINTA2, linewidth=0.9))
for caixa_i, f in zip(cx['boxes'], fams):
    caixa_i.set(facecolor=CORES[f], alpha=0.18, edgecolor=CORES[f], linewidth=1.2)
rng = np.random.default_rng(20260826)
for i_f, (f, d) in enumerate(zip(fams, dados), start=1):
    ax.scatter(rng.normal(i_f, 0.045, len(d)), d, s=26, color=CORES[f],
               edgecolor='white', linewidth=0.8, zorder=3)
ax.set_xticks(range(1, len(fams) + 1))
ax.set_xticklabels([f'{ROTULO_EN[f]}\n(n = {len(d)})' for f, d in zip(fams, dados)])
ax.set_ylabel('Area under the ROC curve')
ax.set_title('Discrimination by outcome, best model per combination')
ax.axhline(0.5, color=TINTA2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.text(0.62, 0.505, 'chance level', va='bottom', ha='left', fontsize=7.5, color=TINTA2)
ax.set_ylim(0.45, 1.03)
ax.set_axisbelow(True)
ax.yaxis.grid(True)
fig_salva(fig, 'figS1_auc_desfecho')
plt.show()

## 08. Class imbalance and heterogeneity

Outcome prevalence computed from events over denominator (CHARMS 5). The heterogeneity table consolidates, by outcome, the differences in prevalence, sample, country and validation that justify a narrative synthesis without meta-analysis (answer to R2.8).

In [ ]:
# -- 08.1 prevalence --
prev = df_comb_eleg['outcome_prevalence'].dropna()
print(f'prevalence reportable: {len(prev)}/{n_comb_eleg} combinations ({100*len(prev)/n_comb_eleg:.0f}%) '
      f'| manuscript: 37/47 (79%)')
print(f'range {100*prev.min():.1f}% to {100*prev.max():.1f}%, median {100*prev.median():.1f}% '
      f'(IQR {100*prev.quantile(.25):.1f}-{100*prev.quantile(.75):.1f}) | manuscript: 0.7-79.5%, median 17.0%')
estratos = pd.cut(prev, [0, .1, .3, 1], labels=['<10%', '10-30%', '>30%']).value_counts().sort_index()
print('strata:', dict(estratos), '| manuscript: 12/14/11')
por_fam = df_comb_eleg.groupby('outcome_family')['outcome_prevalence'].median().dropna().mul(100).round(1)
print('median by outcome (%):', dict(por_fam))
desb = df_comb_eleg[['ID', 'outcome_family', 'N Events', 'outcome_prevalence', 'Imbalance Handling']].copy()
grava(desb.sort_values(['outcome_family', 'ID']), 'class_imbalance.csv')

In [ ]:
# -- 08.2 heterogeneity table --
het = (df_comb_eleg.groupby('outcome_family')
       .agg(combinacoes=('ID', 'count'),
            N_min=('N (sample)', 'min'), N_max=('N (sample)', 'max'), N_mediano=('N (sample)', 'median'),
            prev_min=('outcome_prevalence', 'min'), prev_max=('outcome_prevalence', 'max'),
            paises=('country', 'nunique'),
            pct_indonesia=('country', lambda s: round(100 * (s == 'Indonesia').mean())),
            tipos_validacao=('validation', 'nunique'))
       .reset_index())
for c in ['prev_min', 'prev_max']:
    het[c] = (100 * het[c]).round(1)
print(het.to_string(index=False))
grava(het, 'heterogeneity.csv')

## 09. Model purpose and prediction horizon

The critical path of the review (T1.3/T1.4; demands R2.7 and R2.11): to distinguish diagnostic classification (nowcasting, predictors contemporaneous with the outcome) from prospective prediction (predictors measured before the outcome). The automatic proposal uses `Predictor Timing` from the CHARMS extraction; the anthropometry flag uses `3 Most Important Features` and `Predictor Types`/`Predictor Notes` with terms of the same dimension as the outcome.

The rules below were corrected on 24 Aug 2026 after the audit against the PDFs (section 13). Four corrections: `before or at the outcome` is no longer read as prospective (the "at" is already contemporaneous); the name of the outcome itself is removed from the text before the term search (otherwise "underweight" matches "weight"); parental anthropometry no longer counts as circularity with the child's outcome; and a measurement at birth is now treated as a baseline predictor, not as a contemporaneous measurement. A wider flag, `antro_crianca_qualquer`, was added for cases where the child's anthropometry is of a dimension other than that of the outcome.

The product is a case-by-case template for IVS to check (`rotulo_IVS`, `leakage_IVS`); the notebook never overwrites a template that already carries marks.

In [ ]:
# -- 09.1 automatic label and outcome-anthropometry flag --
def rotulo_finalidade(timing):
    t = str(timing).lower()
    if 'before or at' in t or t.startswith('mixed'):
        return 'mixed'      # 'at the outcome' is already contemporaneous: it is not prospective
    if 'before the outcome' in t:
        return 'prospective prediction'
    if 'same' in t or 'cross-sectional' in t or 'concurrent' in t:
        return 'diagnostic classification (nowcasting)'
    return 'unclear'

# Three corrections over the version of 19 Aug, all confirmed by the PDF audit of 24 Aug:
# (1) the name of the outcome itself is removed from the text before the search, otherwise 'underweight' matches 'weight';
# (2) PARENTAL anthropometry (maternal BMI, mother's height) does not create circularity with the
#     child's outcome, and it was what triggered the flag in AA_07, AA_45, AA_54, AA_58 and AA_61;
# (3) a measurement at BIRTH is a baseline predictor, not a measurement contemporaneous with the outcome.
NOMES_DESFECHO = (r"\b(under\s*weight|underweight|over\s*weight|overweight|stunting|stunted|stunt|"
                  r"wasting|wasted|obesity|obese|thinness|thin|malnutrition|malnourished|"
                  r"nutritional status|nutrition level)\b")
PARENTAL = (r"\b(maternal|mother'?s?|paternal|father'?s?|parent(?:al|s)?|famil(?:y|ial)|sibling|"
            r"pre-?pregnancy|gestational)\b[^,;.]{0,30}?\b(bmi|body mass|weight|height|stature|imc)\b")
# free suffix after the measure: 'birth_weight_3' is also a measurement at birth
NASCIMENTO = r"\b(birth|born|neonatal)[\s_-]*(weight|length|height|size|bmi)\w*|\b(weight|length|height|size)\s+at\s+birth\b"

# terms of the SAME dimension as the outcome (circularity) and terms of child anthropometry in general
TERMOS = {'stunting': [r'height', r'length', r'stature', r'\bhaz\b', r'tb\s*/\s*u', r'\bh\s*/\s*a\b',
                       r'height[- ]for[- ]age'],
          'obesity': [r'weight', r'\bbmi\b', r'\bimc\b', r'\bzbmi\b', r'body mass', r'\bbaz\b'],
          'overweight': [r'weight', r'\bbmi\b', r'\bimc\b', r'\bzbmi\b', r'body mass', r'\bbaz\b'],
          'overweight/obesity (combined)': [r'weight', r'\bbmi\b', r'\bimc\b', r'\bzbmi\b', r'body mass'],
          'thinness': [r'weight', r'\bbmi\b', r'\bwhz\b', r'\bbaz\b'],
          'wasting': [r'weight', r'\bwhz\b', r'weight[- ]for[- ](height|length)', r'\bbmi\b'],
          'underweight': [r'weight', r'\bwaz\b', r'weight[- ]for[- ]age'],
          'malnutrition (composto)': [r'weight', r'height', r'\bbmi\b', r'z[- ]?score']}
ANTRO_CRIANCA = [r'weight', r'height', r'length', r'stature', r'\bbmi\b', r'\bimc\b', r'body mass',
                 r'\bhaz\b', r'\bwaz\b', r'\bwhz\b', r'\bbaz\b', r'\bzbmi\b',
                 r'(mid[- ])?upper[- ]arm circumference', r'\bmuac\b', r'head circumference',
                 r'tb\s*/\s*u', r'\bw\s*/\s*h\b', r'\bw\s*/\s*a\b', r'\bh\s*/\s*a\b']

def limpa(texto):
    t = str(texto).lower()
    for pat in (NOMES_DESFECHO, PARENTAL, NASCIMENTO):
        t = re.sub(pat, ' ', t)
    return t

def acha_termos(texto, fam, lista=None):
    t = limpa(texto)
    achados = set()
    for pat in (lista if lista is not None else TERMOS.get(fam, [])):
        m = re.search(pat, t)
        if m:
            achados.add(m.group(0).strip())
    return sorted(achados)

top3 = df_comb['3 Most Important Features'].fillna('NR')
pred_txt = df_comb['Predictor Types'].astype(str) + ' ' + df_comb['Predictor Notes'].astype(str)
df_comb['proposed_label'] = df_comb['Predictor Timing'].map(rotulo_finalidade)
df_comb['top3_terms'] = [', '.join(acha_termos(v, f)) for v, f in zip(top3, df_comb['outcome_family'])]
df_comb['predictor_terms'] = [', '.join(acha_termos(v, f)) for v, f in zip(pred_txt, df_comb['outcome_family'])]
df_comb['anthropometry_in_top3'] = df_comb['top3_terms'] != ''
df_comb['anthropometry_among_predictors'] = (df_comb['predictor_terms'] != '') | df_comb['anthropometry_in_top3']
# wider flag: any contemporaneous child anthropometry, even of a different dimension
df_comb['any_child_anthropometry'] = [bool(acha_termos(a + ' ' + b, f, ANTRO_CRIANCA))
                                     for a, b, f in zip(pred_txt, top3, df_comb['outcome_family'])]
print(df_comb.groupby('proposed_label')['ID'].nunique().rename('studies').to_string())
print()
print(pd.crosstab(df_comb['proposed_label'], df_comb['anthropometry_in_top3'],
                  rownames=['model purpose'], colnames=['outcome anthropometry in top-3']).to_string())

In [ ]:
# -- 09.2 checking template (anti-overwrite) --
cols = ['ID', 'First Author', 'Year', 'outcome_family', 'eligible', 'Outcome', 'Predictor Timing',
        '3 Most Important Features', 'proposed_label', 'top3_terms', 'predictor_terms',
        'anthropometry_in_top3', 'anthropometry_among_predictors', 'any_child_anthropometry']
IVS_COLS = ['rotulo_IVS', 'leakage_IVS', 'obs_IVS']
template = df_comb[cols].sort_values(['outcome_family', 'ID']).copy()
for c in IVS_COLS:
    template[c] = ''
if os.path.exists(TEMPLATE):
    # anti-overwrite: the IVS columns are never touched. The automatic columns ARE
    # updated, because the rules changed on 24 Aug (section 13) and an obsolete proposed label
    # next to a mark left by IVS is worse than none.
    atual = pd.read_csv(TEMPLATE, sep=';', encoding='utf-8-sig')
    chave = atual['ID'] + '|' + atual['outcome_family']
    marcas = atual.set_index(chave)[IVS_COLS].fillna('')
    novo = template.copy()
    novo.index = novo['ID'] + '|' + novo['outcome_family']
    for c in IVS_COLS:
        novo[c] = marcas[c].reindex(novo.index).fillna('')
    n_marcadas = int(sum((marcas[c] != '').sum() for c in ['rotulo_IVS', 'leakage_IVS']))
    antigo_rot = atual.set_index(chave)['proposed_label'].reindex(novo.index)
    mudou = int((antigo_rot.fillna('') != novo['proposed_label']).sum())
    novas = int((~novo.index.isin(chave)).sum())
    novo = novo.reset_index(drop=True)
    novo.to_csv(TEMPLATE, sep=';', encoding='utf-8-sig', index=False)
    template_uso = novo
    print(f'template updated: {n_marcadas} marks by IVS preserved; {novas} new combinations; '
          f'{mudou} automatic labels changed under the corrected rules')
else:
    template.to_csv(TEMPLATE, sep=';', encoding='utf-8-sig', index=False)
    template_uso = template
    print(f'template created: resultados/classificacao_finalidade_TEMPLATE.csv ({len(template)} combinations) - awaiting the check by IVS')

## 10. Leakage recounted

Two scenarios over the eligible combinations: (a) the definition in the current manuscript (any anthropometry related to the outcome among the predictors = leakage; it was 22/47, 47%); (b) the definition conditioned on model purpose, answering R2.11: in a diagnostic classification model the use of contemporaneous anthropometry is diagnostic tautology (the model recovers the outcome itself), and genuine leakage is reserved for prospective models using anthropometry contemporaneous with the outcome. It uses `rotulo_IVS`/`leakage_IVS` once IVS has checked the template; otherwise it uses the automatic proposal, with a warning. It also rebuilds the performance contrast (0.97 vs 0.85 in the text) and the cross-tabulation with the PROBAST predictors domain (making the 73%/8% of the Discussion traceable).

In [ ]:
# -- 10.1 cenarios --
uso = template_uso.copy()
uso['final_label'] = np.where(uso['rotulo_IVS'].fillna('') != '', uso['rotulo_IVS'], uso['proposed_label'])
if (uso['rotulo_IVS'].fillna('') == '').all():
    print('WARNING: template not yet checked by IVS; using the automatic labels.')
el = uso[uso['eligible']].copy()
nA = int(el['anthropometry_among_predictors'].sum())
nQ = int(el['any_child_anthropometry'].sum())
diag = el['final_label'].str.contains('diagnostic')
taut = int((el['anthropometry_among_predictors'] & diag).sum())
prosp = el['final_label'].str.contains('prospective')
leak_genuino = el[prosp & el['anthropometry_among_predictors']]
print(f'scenario (a), current definition: {nA}/{len(el)} combinations with anthropometry of the SAME dimension as the '
      f'outcome among the predictors ({100*nA/len(el):.0f}%) | manuscript: 22/47 (47%)')
print(f'  wider flag (any contemporaneous child anthropometry): {nQ}/{len(el)} ({100*nQ/len(el):.0f}%)')
print(f'scenario (b): {int(diag.sum())}/{len(el)} combinations are diagnostic classification ({100*diag.mean():.0f}%); '
      f'of these, {taut} use anthropometry of the outcome itself (diagnostic tautology)')
print(f'candidates for genuine leakage (prospective + outcome anthropometry): {len(leak_genuino)} '
      f'{list(leak_genuino["ID"]) if len(leak_genuino) else ""}')
print('   the PDF audit (section 13) rules both out: see the note below - the decision goes in leakage_IVS')
print(f'prospective models: {int(prosp.sum())} combinations, studies {sorted(el.loc[prosp, "ID"].unique())}')

### Note on the two candidates for genuine leakage

The automatic rule flags as leakage every prospective model that uses anthropometry of the outcome itself. The audit against the PDFs (section 13) shows that neither case holds, and the distinction is exactly the one Reviewer 2 asks for in comment R2.11:

- **AA_31**: the predictor anthropometry (weight and weight-for-height) is measured from 0 to 24 months and the outcome is obesity between 2 and 7 years, defined by the CDC BMI percentile, which applies only from age 2. It is a baseline measurement, prior to the outcome, with no overlap of windows.
- **AA_50**: the BMI z-score in kindergarten is the rank-1 predictor in all four algorithms, but it precedes the outcome (obesity in grade 4) by about four school years and is the study's own exposure of interest, which quantifies the informational value of school screening. The article measures the effect: without the z-score the AUC falls to 0.512; with it alone it already reaches 0.782, against 0.781 to 0.785 for the full model.

An anthropometric measurement taken before the outcome, with a declared prediction horizon, is a legitimate predictor. What the current version of the manuscript calls leakage is, in cross-sectional models, diagnostic tautology: the model recovers the outcome from the variables that define it. The final decision stays in the `leakage_IVS` column of the template.

In [ ]:
# -- 10.2 performance contrast and PROBAST x predictors --
mapa_flag = dict(zip(el['ID'] + '|' + el['outcome_family'], el['anthropometry_among_predictors']))
dce = df_comb_eleg.copy()
dce['anthropometry_flag'] = (dce['ID'] + '|' + dce['outcome_family']).map(mapa_flag)
for m in ['Accuracy', 'AUC']:
    a = dce.loc[dce['anthropometry_flag'] == True, m].dropna()
    b = dce.loc[dce['anthropometry_flag'] == False, m].dropna()
    print(f'{m}: median with outcome anthropometry {a.median():.2f} (n={len(a)}) vs without {b.median():.2f} (n={len(b)})'
          + (' | manuscript: 0.97 vs 0.85' if m == 'Accuracy' else ''))
d2 = pd.crosstab(dce['anthropometry_flag'], dce['Dev D2 Predictors'].astype(str).str.strip().str.title(), normalize='index').mul(100).round(0)
print('\nPROBAST Dev D2 Predictors (% per row, eligible combinations):')
print(d2.to_string())
print('(the 73%/8% figures in the current Discussion need to be replaced by the cross-tabulation above)')
rec = el[['ID', 'outcome_family', 'final_label', 'anthropometry_in_top3', 'anthropometry_among_predictors', 'top3_terms']].copy()
rec = rec.merge(dce[['ID', 'outcome_family', 'AUC', 'Accuracy']], on=['ID', 'outcome_family'], how='left')
grava(rec.sort_values(['outcome_family', 'ID']), 'leakage_classification.csv')

In [ ]:
# -- 10.3 Figure 4: most frequent predictors in the top-3 --
# canonical names: the free text of the articles carries variants of the same variable
CANONICO = [
    (r'weight[- ]for[- ](height|length)|\bwhz\b', 'Weight-for-height'),
    (r'weight[- ]for[- ]age|\bwaz\b', 'Weight-for-age'),
    (r'height[- ]for[- ]age|\bhaz\b', 'Height-for-age'),
    (r'birth\s*(weight|_weight)|weight at birth', 'Birth weight'),
    (r'birth\s*(length|height)', 'Birth length'),
    (r"(maternal|mother'?s?)\s*(bmi|body mass)", "Maternal BMI"),
    (r"(maternal|mother'?s?)\s*(height|stature)", "Maternal height"),
    (r"(maternal|mother'?s?|mother)\s*educ", 'Maternal education'),
    (r"(paternal|father'?s?|partner)\s*educ", 'Paternal education'),
    (r'wealth', 'Household wealth'),
    (r'\bregion\b|province|district', 'Region'),
    (r'child.*sex|^sex|gender', 'Sex'),
    (r'child.*age|^age|age in months', 'Age'),
    (r'child.*(length|height)|^height|^length|stature', 'Height or length'),
    (r'child.*weight|^weight|weight at ', 'Weight'),
    (r'\bbmi\b|body mass', 'Body mass index'),
    (r'muac|upper arm circumference', 'MUAC'),
    (r'antenatal', 'Antenatal care'),
    (r'place of delivery|delivery place', 'Place of delivery'),
    (r'breastfeed|breast feeding', 'Breastfeeding'),
    (r'consanguineous', 'Consanguineous marriage'),
    (r'television|\btv\b', 'Television ownership'),
    (r'fast food', 'Fast-food intake'),
    (r'toilet|sanitation|water source', 'Water and sanitation'),
    (r'residen|urban|rural', 'Place of residence'),
    (r'diarrh', 'Diarrhoea'),
    (r'vitamin|supplement', 'Micronutrient supplementation'),
    (r'household size|number of (children|household)', 'Household size'),
]


def canoniza(txt):
    t = str(txt).strip().lower()
    for padrao, nome in CANONICO:
        if re.search(padrao, t):
            return nome
    return t[:38].capitalize()


top = df_comb_eleg[['outcome_family', '3 Most Important Features']].dropna()
top = top[top['3 Most Important Features'].str.upper() != 'NR']
feats = []
for fam, linha in zip(top['outcome_family'], top['3 Most Important Features']):
    vistos = set()
    for f in str(linha).split(';'):
        f = f.strip()
        if not f:
            continue
        nome = canoniza(f)
        if nome in vistos:      # variants of the same variable count once per combination
            continue
        vistos.add(nome)
        feats.append({'predictor': nome, 'outcome_dimension_anthropometry': bool(acha_termos(f, fam))})
fdf = pd.DataFrame(feats)
cont = (fdf.groupby('predictor')
        .agg(combinacoes=('predictor', 'count'), antro_desfecho=('outcome_dimension_anthropometry', 'max'))
        .sort_values('combinations'))
grava(cont.reset_index().sort_values('combinations', ascending=False), 'figure4_predictors.csv')
cont = cont.tail(15)

VERM, AZUL = '#C2453A', '#2A6FB0'
fig, ax = plt.subplots(figsize=(7.5, 5.2))
cores_barra = [VERM if a else AZUL for a in cont['outcome_dimension_anthropometry']]
barras = ax.barh(cont.index, cont['combinations'], color=cores_barra, height=0.68)
rotula_barras(ax, barras, cont['combinations'].values, dx=0.15, horizontal=True)
ax.set_xlabel(f'Study-outcome combinations reporting the predictor among the three most important '
              f'(n = {len(top)})')
ax.set_xlim(0, cont['combinations'].max() * 1.12)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_axisbelow(True)
ax.xaxis.grid(True)
mp = [plt.Rectangle((0, 0), 1, 1, color=VERM), plt.Rectangle((0, 0), 1, 1, color=AZUL)]
ax.legend(mp, ['Anthropometry sharing the outcome dimension', 'Other predictors'],
          loc='lower right', handlelength=1.1, handleheight=1.1)
fig_salva(fig, 'fig4_preditores')
plt.show()
print(f'combinations with a reported top-3: {len(top)}/{n_comb_eleg} | canonical predictors: {fdf["predictor"].nunique()}')

In [ ]:
# -- 10.4 Figure S2: performance by type of predictor --
sub = df_comb[df_comb['eligible']].copy()  # carries the flags from section 09.1
sub['group'] = np.where(sub['anthropometry_among_predictors'], 'Outcome-dimension\nanthropometry', 'Other predictors')
fig, eixos = plt.subplots(1, 2, figsize=(8.5, 4.2), sharey=True)
for eixo, metrica, titulo in zip(eixos, ['Accuracy', 'AUC'],
                                 ['A. Accuracy', 'B. Area under the ROC curve']):
    grupos = ['Outcome-dimension\nanthropometry', 'Other predictors']
    dados = [pd.to_numeric(sub.loc[sub['group'] == g, metrica], errors='coerce').dropna() for g in grupos]
    cx = eixo.boxplot(dados, widths=0.4, showfliers=False, patch_artist=True,
                      medianprops=dict(color=TINTA, linewidth=1.6),
                      whiskerprops=dict(color=TINTA2, linewidth=0.9),
                      capprops=dict(color=TINTA2, linewidth=0.9))
    for caixa_i, cor in zip(cx['boxes'], [VERM, AZUL]):
        caixa_i.set(facecolor=cor, alpha=0.18, edgecolor=cor, linewidth=1.2)
    rng2 = np.random.default_rng(20260826)
    for i_g, (d, cor) in enumerate(zip(dados, [VERM, AZUL]), start=1):
        eixo.scatter(rng2.normal(i_g, 0.045, len(d)), d, s=26, color=cor,
                     edgecolor='white', linewidth=0.8, zorder=3)
        eixo.text(i_g, 0.565, f'n = {len(d)}\nmedian {d.median():.2f}', ha='center', va='bottom',
                  fontsize=8, color=TINTA2)
    eixo.set_xticks([1, 2])
    eixo.set_xticklabels(grupos)
    eixo.set_title(titulo)
    eixo.set_ylim(0.55, 1.04)
    eixo.set_axisbelow(True)
    eixo.yaxis.grid(True)
eixos[1].spines['left'].set_visible(False)
eixos[0].set_ylabel('Reported value')
fig_salva(fig, 'figS2_desempenho_preditor')
plt.show()

## 11. PROBAST+AI by domain

Demand R2.6: Reviewer 2 read the aggregate ("96% high risk") as low discriminatory capacity of the instrument. The answer is the decomposition by domain (D1 to D4, development and evaluation), which shows variation between domains. Denominator: 38 studies. The numbers match those already published in the Doc (97%, 87%/92%, applicability 61%/58%).

In [ ]:
# -- 11.1 decomposition by domain --
DOM = {'Dev D1 Participants': 'D1 participants', 'Dev D2 Predictors': 'D2 predictors',
       'Dev D3 Outcomes': 'D3 outcome', 'Dev D4 Analyses': 'D4 analysis',
       'Dev Overall': 'overall', 'Dev Applicability': 'applicability',
       'Eval D1 Participants': 'D1 participants', 'Eval D2 Predictors': 'D2 predictors',
       'Eval D3 Outcomes': 'D3 outcome', 'Eval D4 Apparent': 'D4 apparent',
       'Eval D4 Internal': 'D4 internal', 'Eval D4 External': 'D4 external',
       'Eval Overall': 'overall', 'Eval Applicability': 'applicability'}
blocos = []
for col, rot in DOM.items():
    v = df_est[col].astype(str).str.strip().str.title().replace({'Nan': 'NA'})
    c = v.value_counts()
    blocos.append({'part': 'development' if col.startswith('Dev') else 'evaluation', 'domain': rot,
                   **{k: int(c.get(k, 0)) for k in ['Low', 'High', 'Unclear', 'Na']}})
prob = pd.DataFrame(blocos)
for k in ['Low', 'High', 'Unclear']:
    prob[f'{k}_pct'] = (100 * prob[k] / 38).round(0).astype(int)
print(prob.to_string(index=False))
alto_dev = prob.loc[(prob['part'] == 'development') & (prob['domain'] == 'overall'), 'High'].iloc[0]
alto_ev = prob.loc[(prob['part'] == 'evaluation') & (prob['domain'] == 'overall'), 'High'].iloc[0]
print(f'\ngeral: desenvolvimento {alto_dev}/38 High; avaliacao {alto_ev}/38 High | texto atual: 37/38 (97%) nas duas partes')
grava(prob, 'probastai_domains_intermediate.csv')

## 12. Numbers for the manuscript

Consolidation: one row per numerical statement in the current text, with the old value (base 43/47) and the new value (base v7). It is the guide for the substitutions in the Google Doc. The values in the leakage sections must be rerun after IVS has checked the template.

In [ ]:
# -- 12.1 consolidacao --
fmt = lambda v: ('' if pd.isna(v) else (f'{v:.2f}' if isinstance(v, float) else str(v)))
def med_iqr(serie):
    v = serie.dropna()
    return f'{v.median():.2f} (IQR {v.quantile(.25):.2f}-{v.quantile(.75):.2f}; n={len(v)})' if len(v) else 'NR'

L = []
def add(secao, afirmacao, antigo, novo):
    L.append({'section': secao, 'statement': afirmacao, 'value_in_current_manuscript': antigo, 'new_value_v7': str(novo)})

add('Abstract/Results', 'included studies', '43 studies / 47 combinations',
    f'{n_est_eleg} studies / {n_comb_eleg} eligible combinations (38 studies in the full base)')
for fam in ELEGIVEIS:
    n = int((df_comb_eleg['outcome_family'] == fam).sum())
    add('Study characteristics', f'combinations of {fam}', f"{ANTIGO.get(fam, 0)}/47",
        f'{n}/{n_comb_eleg} ({100*n/n_comb_eleg:.0f}%)' if n else '0 (outcome absent from the base of 38)')
add('Study characteristics', 'countries', '15 countries', f'{n_paises} paises')
add('Study characteristics', 'Indonesia', '23/47 (49%)',
    f'{ind_e}/38 studies ({100*ind_e/38:.0f}%); {ind_c}/{n_comb_eleg} combinations ({100*ind_c/n_comb_eleg:.0f}%)')
add('Study characteristics', 'pico de publicacao', '26/47 (55%) em 2024',
    f'{int(anos.max())}/38 studies ({100*anos.max()/38:.0f}%) in {pico}')
add('Study characteristics', 'amostra', '114 a 244.053 (mediana 6.677)',
    f'{int(ns.min())} a {int(ns.max())} (mediana {int(ns.median())}; {len(ns)}/38 com N)')
add('Study characteristics', 'Random Forest', '16/47 (34%)',
    f'{rf_est}/38 studies ({100*rf_est/38:.0f}%); best model in {rf_melhor}/{n_comb_eleg} '
    f'combinations ({100*rf_melhor/n_comb_eleg:.0f}%); {rf}/241 rows')
add('Study characteristics', 'deep learning', 'two studies', f'{len(dl_est)} studies')
add('Predictive performance', 'AUC reported', '20/47 (43%)', f'{auc_rep}/{n_comb_eleg} ({100*auc_rep/n_comb_eleg:.0f}%)')
for fam in ELEGIVEIS:
    g = df_comb_eleg[df_comb_eleg['outcome_family'] == fam]
    if len(g):
        add('Predictive performance', f'AUC {fam}', 'see the text', med_iqr(g['AUC']))
        add('Predictive performance', f'Accuracy {fam}', 'see the text', med_iqr(g['Accuracy']))
add('Predictive performance', 'external validation', 'one study (2%)',
    f"{ext['ID'].nunique()} estudo(s): {list(ext['ID'].unique())}")
add('Class imbalance', 'prevalencia reportavel', '37/47 (79%)',
    f'{len(prev)}/{n_comb_eleg} ({100*len(prev)/n_comb_eleg:.0f}%)')
add('Class imbalance', 'prevalencia mediana', '17.0% (IQR 7.3-35.4)',
    f'{100*prev.median():.1f}% (IQR {100*prev.quantile(.25):.1f}-{100*prev.quantile(.75):.1f})')
add('Predictor variables', 'outcome anthropometry (current definition)', '22/47 (47%)',
    f'{nA}/{len(el)} ({100*nA/len(el):.0f}%) [rerodar apos conferencia IVS]')
add('Predictor variables', 'contraste de accuracy', '0.97 vs 0.85',
    f"{dce.loc[dce['anthropometry_flag'] == True, 'Accuracy'].median():.2f} vs {dce.loc[dce['anthropometry_flag'] == False, 'Accuracy'].median():.2f} [rerodar apos conferencia IVS]")
add('Discussion', 'diagnostic classification vs prediction', 'not classified in the text',
    f'{int(diag.sum())}/{len(el)} nowcasting combinations ({100*diag.mean():.0f}%); prospective: {int(prosp.sum())}')
add('Risk of bias', 'PROBAST overall', '37/38 (97%) High in both parts',
    f'dev {alto_dev}/38 High; eval {alto_ev}/38 High')
if os.path.exists(AUDITORIA):
    a_ = pd.read_csv(AUDITORIA, sep=';', encoding='utf-8-sig')
    d_ = pd.read_csv(DIVERGENCIAS, sep=';', encoding='utf-8-sig')
    sev = d_['severidade'].value_counts()
    add('Methods/Limitations', 'audit of the extraction', 'not reported in the manuscript',
        f"{len(a_)}/38 studies re-audited against the PDF (24 Aug); {sev.get('alta', 0)} high divergences, "
        f"{sev.get('media', 0)} medium, {sev.get('baixa', 0)} low; no performance value in the wrong row")
    add('Predictor variables', 'vazamento genuino', '22/47 (47%) descritos como data leakage',
        'nenhum dos 3 modelos prospectivos apos auditoria: AA_31 e AA_50 usam antropometria anterior ao '
        'outcome, with a prediction horizon; the rest is diagnostic tautology in a cross-sectional model')
    add('Predictive performance', 'comparability of the metrics', 'not qualified in the manuscript',
        'precision/recall/F1 are macro-average or weighted average, and not the metric of the positive class, '
        'in part of the studies (see auditoria_v7_temas.csv) - they should not be pooled without a caveat')
numeros = pd.DataFrame(L)
print(numeros.to_string(index=False))
grava(numeros, 'numeros_manuscrito.csv')

## 13. Audit of the extraction against the PDFs (24 Aug 2026)

Each of the 38 studies was re-audited by an independent reader who opened only its PDF and checked, field by field, what v7 states: identification, outcomes, predictors and prediction horizon, algorithms, validation, the six performance metrics row by row, events and calibration. Rules of the audit: a value stated in the abstract is not a valid source, whatever the article does not report is NR and whatever the auditor cannot locate is NAO_VERIFICAVEL, and every divergence needs a page and a table.

This section reads the consolidated result. It measures two different things: the fidelity of the transcription (does the spreadsheet reproduce the article?) and the coherence of what was extracted (does the set make sense given the study design?).

In [ ]:
# -- 13.1 verdicts by block --
if not os.path.exists(AUDITORIA):
    print('audit not consolidated yet; run audit_v7/consolida.py before this section')
else:
    aud = pd.read_csv(AUDITORIA, sep=';', encoding='utf-8-sig')
    div = pd.read_csv(DIVERGENCIAS, sep=';', encoding='utf-8-sig')
    BLOCOS = ['A_identificacao', 'B_desfecho', 'C_preditores', 'D_modelo',
              'E_validacao', 'F_desempenho', 'G_eventos', 'H_calibracao']
    print(f'studies audited: {len(aud)}/38')
    vb = pd.DataFrame({b: aud[b].value_counts() for b in BLOCOS}).T.fillna(0).astype(int)
    for c in ['OK', 'PARCIAL', 'DIVERGE', 'NAO_VERIFICAVEL']:
        if c not in vb.columns:
            vb[c] = 0
    vb = vb[['OK', 'PARCIAL', 'DIVERGE', 'NAO_VERIFICAVEL']]
    print()
    print(vb.to_string())
    grava(vb.reset_index().rename(columns={'index': 'block'}), 'verification_verdicts_by_block.csv')

In [ ]:
# -- 13.2 divergences by severity and block --
if os.path.exists(DIVERGENCIAS):
    print(div['severidade'].value_counts().rename('divergences').to_string())
    print()
    print(pd.crosstab(div['block'], div['severidade']).to_string())
    altas = div[div['severidade'] == 'alta']
    print(f'\n== {len(altas)} divergences of high severity ==')
    for _, r in altas.iterrows():
        print(f"{r['ID']} | {r['campo']}: spreadsheet '{str(r['valor_na_planilha'])[:28]}' vs article "
              f"'{str(r['valor_no_artigo'])[:28]}' ({r['evidencia']})")
    lim = aud[aud['n_divergences'] == 0]['ID'].tolist()
    print(f'\nstudies with no divergence at all: {len(lim)} {lim}')

In [ ]:
# -- 13.3 model purpose: auditor vs automatic proposal --
if os.path.exists(AUDITORIA):
    conc = aud['auditor_concorda'].astype(str).str.lower()
    print(f"auditor agrees with the model-purpose label: {int((conc == 'true').sum())}/{len(aud)}")
    disc = aud[conc == 'false']
    for _, r in disc.iterrows():
        print(f"  {r['ID']}: proposed '{r['finalidade_proposta']}' -> auditor '{r['finalidade_correta']}'")
    print()
    antro = aud['antropometria_do_desfecho_entre_preditores'].astype(str).str.lower()
    print(f"auditors confirm anthropometry of the outcome itself among the predictors in "
          f"{int((antro == 'true').sum())}/{len(aud)} studies")

In [ ]:
# -- 13.4 coherence reading by study --
if os.path.exists(AUDITORIA):
    cols_ver = ['ID', 'article', 'design', 'n_divergences', 'n_high',
                'finalidade_correta', 'antropometria_do_desfecho_entre_preditores', 'coerencia_geral']
    print(aud[cols_ver].head(38).to_string(index=False, max_colwidth=70))

## 14. Updated search in Embase and Web of Science (R2.4)

Reviewer 2 points out the absence of Embase and Web of Science. The search was repeated in both databases on 9 July 2026 with the original strategy (`base_dados/embase_revision_20260709.ris` and `wos_revision_20260709.ris`).

Three questions are answered here: how many records the two databases returned, how many of those had not been retrieved by the four original databases, and how many of the already included studies the two databases manage to retrieve. The last one is the sensitivity test: if the new databases do not retrieve even the studies already in the review, their coverage of this literature is smaller than that of the original databases.

In [ ]:
# -- 14.1 reading the RIS files and counting by database --
import unicodedata

def norm_txt(t):
    t = unicodedata.normalize('NFKD', str(t)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', t.lower())).strip()

def le_ris(caminho):
    regs, cur = [], {}
    for ln in open(caminho, encoding='utf-8', errors='ignore'):
        m = re.match(r'^([A-Z][A-Z0-9])  - (.*)$', ln.rstrip('\n'))
        if not m:
            continue
        tag, val = m.groups()
        if tag == 'TY':
            if cur:
                regs.append(cur)
            cur = {}
        cur.setdefault(tag, []).append(val)
    if cur:
        regs.append(cur)
    saida = []
    for r in regs:
        ti = (r.get('TI') or r.get('T1') or [''])[0]
        saida.append({'titulo': ti, 'doi': (r.get('DO') or [''])[0].lower().strip(),
                      'ano': (r.get('Y1') or r.get('PY') or [''])[0][:4], 'tnorm': norm_txt(ti)})
    return saida

emb = le_ris(f'{BASE_DADOS}/embase_revision_20260709.ris')
wos = le_ris(f'{BASE_DADOS}/wos_revision_20260709.ris')
print(f'Embase: {len(emb)} registros | Web of Science: {len(wos)} registros | total {len(emb) + len(wos)}')

orig_chaves = set()
for arq in ['sistematica_ml_07_23_2025_bvs.ris', 'sistematica_ml_07_23_2025_scopus.ris',
            'sistematica_ml_07_23_2025_ieee_pt1.ris', 'sistematica_ml_07_23_2025_ieee_pt2.ris']:
    for r in le_ris(f'{BASE_DADOS}/{arq}'):
        orig_chaves.update({k for k in (r['tnorm'], r['doi']) if k})
pm = pd.read_csv(f'{BASE_DADOS}/sistematica_ml_07_23_2025_pubmed.csv')
col_ti = [c for c in pm.columns if 'title' in c.lower()][0]
orig_chaves.update({norm_txt(t) for t in pm[col_ti] if isinstance(t, str)})
if 'DOI' in pm.columns:
    orig_chaves.update({str(d).lower().strip() for d in pm['DOI'] if isinstance(d, str) and d.strip()})
print(f'title/DOI keys from the original search: {len(orig_chaves)}')

In [ ]:
# -- 14.2 deduplication and previously unretrieved records --
unicos = {}
for nome, base in [('Embase', emb), ('Web of Science', wos)]:
    for r in base:
        ch = r['doi'] or r['tnorm']
        if not ch:
            continue
        unicos.setdefault(ch, {'reg': r, 'bases': set()})['bases'].add(nome)
dup_entre_bases = len(emb) + len(wos) - len(unicos)
ineditos = {c: v for c, v in unicos.items()
            if v['reg']['tnorm'] not in orig_chaves and not (v['reg']['doi'] and v['reg']['doi'] in orig_chaves)}
ja_recuperados = len(unicos) - len(ineditos)
print(f'unicos apos deduplicacao entre as duas bases: {len(unicos)} (duplicatas entre bases: {dup_entre_bases})')
print(f'already retrieved by the original search: {ja_recuperados} | not previously retrieved: {len(ineditos)}')

novos_df = pd.DataFrame([{'titulo': v['reg']['titulo'], 'ano': v['reg']['ano'], 'doi': v['reg']['doi'],
                          'bases': ' + '.join(sorted(v['bases'])), 'triagem_IVS': '', 'motivo_IVS': ''}
                         for v in ineditos.values()]).sort_values(['ano', 'titulo'], ascending=[False, True])
grava(novos_df, 'busca_atualizada_novos.csv')

In [ ]:
# -- 14.3 sensitivity test: do the new databases retrieve the already included studies? --
dois_inc = {str(d).lower().strip() for d in df_est['DOI'] if isinstance(d, str) and d.strip() and d.upper() != 'NR'}
rec = {}
for nome, base in [('Embase', emb), ('Web of Science', wos)]:
    rec[nome] = {r['doi'] for r in base if r['doi'] and r['doi'] in dois_inc}
uniao = rec['Embase'] | rec['Web of Science']
print(f'included studies with a DOI: {len(dois_inc)}/{len(df_est)}')
for nome, hit in rec.items():
    print(f'  {nome}: recupera {len(hit)}/{len(dois_inc)} ({100 * len(hit) / len(dois_inc):.0f}%)')
print(f'  union of the two: {len(uniao)}/{len(dois_inc)} ({100 * len(uniao) / len(dois_inc):.0f}%)')
print(f'  four original databases: {len(dois_inc)}/{len(dois_inc)} (100%), by construction of the set')

busca = pd.DataFrame([
    {'base': 'Scopus', 'busca': 'original (05/08/2025)', 'registros': 285, 'fonte_da_contagem': 'documento Estrategia de Busca'},
    {'base': 'PubMed', 'busca': 'original (05/08/2025)', 'registros': 127, 'fonte_da_contagem': 'documento Estrategia de Busca'},
    {'base': 'IEEE Xplore', 'busca': 'original (05/08/2025)', 'registros': 129, 'fonte_da_contagem': 'documento Estrategia de Busca'},
    {'base': 'BVS/LILACS', 'busca': 'original (05/08/2025)', 'registros': 53, 'fonte_da_contagem': 'documento Estrategia de Busca'},
    {'base': 'Embase', 'busca': 'atualizada (09/07/2026)', 'registros': len(emb), 'fonte_da_contagem': 'RIS exportado'},
    {'base': 'Web of Science', 'busca': 'atualizada (09/07/2026)', 'registros': len(wos), 'fonte_da_contagem': 'RIS exportado'},
])
busca['estudos_incluidos_recuperados'] = [None, None, None, None, len(rec['Embase']), len(rec['Web of Science'])]
grava(busca, 'busca_atualizada.csv')

In [ ]:
# -- 14.4 fluxograma PRISMA 2020 --
import matplotlib.patches as mpatches

PRISMA = [
    ('original', 'identified', 594, 'Scopus 285, PubMed 127, IEEE Xplore 129, BVS 53 (documento Estrategia de Busca, 05/08/2025)'),
    ('original', 'duplicates removed', 127, 'log 2.4; the submitted manuscript records 235, an unresolved divergence'),
    ('original', 'screened', 467, '594 - 127'),
    ('original', 'excluded at screening', 384, 'documento Estrategia de Busca'),
    ('original', 'full text assessed', 83, 'documento Estrategia de Busca'),
    ('original', 'excluded at full text', 20, '83 assessed - 63 extracted; ineligible outcome, undefined age range, no predictive modelling'),
    ('original', 'extracted', 63, 'extraction v3'),
    ('original', 'excluded after extraction', 26, 'extraction v4: no isolable eligible outcome, outcome outside the scope, duplicate cohort, composite outcome with no isolable anthropometric dimension'),
    ('updated', 'identified', len(emb) + len(wos), f'Embase {len(emb)}, Web of Science {len(wos)} (09/07/2026)'),
    ('updated', 'duplicates between the two databases', dup_entre_bases, 'DOI or normalised title'),
    ('updated', 'already retrieved by the original search', ja_recuperados, 'DOI or normalised title against the exports of the four databases'),
    ('updated', 'not previously retrieved, awaiting screening', len(ineditos), 'listed in busca_atualizada_novos.csv'),
    ('final', 'studies included in the review', len(df_est), 'extraction base v7'),
    ('final', 'studies with an eligible outcome', df_comb_eleg['ID'].nunique(), 'stunting, overweight, obesity'),
]
grava(pd.DataFrame(PRISMA, columns=['column', 'box', 'n', 'source']), 'prisma_contagens.csv')
val = {(c, k): n for c, k, n, _ in PRISMA}

def caixa(ax, x, y, w, h, txt, cor='white', tam=8.5, negrito=False):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.006',
                                         linewidth=0.9, edgecolor='#333333', facecolor=cor))
    ax.text(x + w / 2, y + h / 2, txt, ha='center', va='center', fontsize=tam,
            fontweight='bold' if negrito else 'normal')

def seta(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-|>', color='#333333', linewidth=0.9))

fig, ax = plt.subplots(figsize=(11.5, 8.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
LX, RX, LW, EW, H = 0.055, 0.545, 0.25, 0.17, 0.088
caixa(ax, LX, 0.915, LW + EW + 0.02, 0.05, 'Original search (four databases, 05 Aug 2025)', '#DCE6F1', 9.5, True)
caixa(ax, RX, 0.915, LW + EW + 0.02, 0.05, 'Updated search (two databases, 09 Jul 2026)', '#FBE5D6', 9.5, True)
ys = [0.795, 0.655, 0.515, 0.375, 0.235]
esq = [f"Records identified\n(n = {val[('original', 'identified')]})",
       f"Records screened\n(n = {val[('original', 'screened')]})",
       f"Reports assessed for eligibility\n(n = {val[('original', 'full text assessed')]})",
       f"Studies data-extracted\n(n = {val[('original', 'extracted')]})",
       f"Studies retained\n(n = {len(df_est)})"]
esq_lat = [f"Duplicate records removed\n(n = {val[('original', 'duplicates removed')]})",
           f"Records excluded\n(n = {val[('original', 'excluded at screening')]})",
           f"Reports excluded (n = {val[('original', 'excluded at full text')]}):\nineligible outcome,\nundefined age range,\nno predictive model",
           f"Studies excluded (n = {val[('original', 'excluded after extraction')]}):\nno eligible outcome after\nfull data extraction",
           None]
dir_ = [f"Records identified\n(n = {val[('updated', 'identified')]})",
        f"Unique records\n(n = {val[('updated', 'identified')] - val[('updated', 'duplicates between the two databases')]})",
        f"Records not retrieved by the\noriginal search (n = {val[('updated', 'not previously retrieved, awaiting screening')]})",
        "Records screened\n(n = to be completed)",
        "New studies included\n(n = to be completed)"]
dir_lat = [f"Duplicates between the two\ndatabases (n = {val[('updated', 'duplicates between the two databases')]})",
           f"Already retrieved by the\noriginal search (n = {val[('updated', 'already retrieved by the original search')]})",
           None, None, None]
for i, y in enumerate(ys):
    caixa(ax, LX, y, LW, H, esq[i])
    if esq_lat[i]:
        caixa(ax, LX + LW + 0.02, y, EW, H, esq_lat[i], '#F2F2F2', 7)
        seta(ax, LX + LW, y + H / 2, LX + LW + 0.02, y + H / 2)
    caixa(ax, RX, y, LW, H, dir_[i], '#F2F2F2' if i >= 3 else 'white')
    if dir_lat[i]:
        caixa(ax, RX + LW + 0.02, y, EW, H, dir_lat[i], '#F2F2F2', 7)
        seta(ax, RX + LW, y + H / 2, RX + LW + 0.02, y + H / 2)
    if i:
        seta(ax, LX + LW / 2, ys[i - 1], LX + LW / 2, y + H)
        seta(ax, RX + LW / 2, ys[i - 1], RX + LW / 2, y + H)
caixa(ax, 0.20, 0.06, 0.60, 0.08,
      f"Studies included in the review (n = {len(df_est)}); "
      f"eligible study-outcome combinations (n = {len(df_comb_eleg)})", '#E2EFDA', 10, True)
seta(ax, LX + LW / 2, ys[-1], 0.42, 0.14)
seta(ax, RX + LW / 2, ys[-1], 0.58, 0.14)
for rot, y0, y1 in [('Identification', 0.795, 0.88), ('Screening', 0.375, 0.74), ('Included', 0.06, 0.32)]:
    ax.text(0.012, (y0 + y1) / 2, rot, rotation=90, ha='center', va='center', fontsize=9,
            fontweight='bold', color='#555555')
ax.set_title('Figure 1. PRISMA 2020 flow diagram', fontsize=11.5, fontweight='bold', loc='left', pad=14)
fig_salva(fig, 'fig1_prisma')
plt.show()

## 15. Certainty of the evidence (R2.2)

Reviewer 2 points out that GRADE is mentioned without domains or result. There is no consolidated GRADE for reviews of prediction models, so the assessment is declared as adapted: the five domains are judged by outcome, PROBAST+AI enters as an input to the risk-of-bias domain, and the final certainty is reported by outcome.

The judgements are derived from quantities computed in the preceding sections, with no manual entry. Downgrading rule: risk of bias by two levels when 80% or more of the studies for that outcome are at high risk, as very serious risk; inconsistency when the AUC range reaches 0.20; indirectness when half or more of the combinations are diagnostic classification; imprecision when fewer than five combinations report AUC.

In [ ]:
# -- 15.1 certainty table by outcome --
linhas_grade = []
comb_fin = df_comb[df_comb['eligible']].copy()  # carries rotulo_proposto from section 09.1
comb_fin['country'] = comb_fin['Country/Region'].map(pais_norm)
for fam in ELEGIVEIS:
    g = comb_fin[comb_fin['outcome_family'] == fam]
    if g.empty:
        continue
    ids = g['ID'].unique()
    auc = pd.to_numeric(g['AUC'], errors='coerce').dropna()
    alto_rv = int((df_est.loc[df_est['ID'].isin(ids), 'Dev Overall'].astype(str).str.strip().str.lower() == 'high').sum())
    nowc = int(g['proposed_label'].astype(str).str.startswith('diagnostic').sum())
    ind_pct = 100 * (g['country'] == 'Indonesia').mean()
    rebaixa, niveis = [], 0
    if alto_rv / len(ids) >= 0.8:
        rebaixa.append('risk of bias (very serious, 2 levels)')
        niveis += 2
    if len(auc) > 1 and (auc.max() - auc.min()) >= 0.20:
        rebaixa.append('inconsistency')
        niveis += 1
    if nowc / len(g) >= 0.5:
        rebaixa.append('indirectness')
        niveis += 1
    if len(auc) < 5:
        rebaixa.append('imprecision')
        niveis += 1
    certeza = {0: 'high', 1: 'moderate', 2: 'low'}.get(niveis, 'very low')
    linhas_grade.append({
        'outcome': fam, 'studies': len(ids), 'combinations': len(g),
        'risk_of_bias': f'high in {alto_rv}/{len(ids)} studies (PROBAST+AI, development)',
        'inconsistency': (f'AUC from {auc.min():.2f} to {auc.max():.2f} in {len(auc)} combinations'
                           if len(auc) else 'AUC not reported in any combination'),
        'indirectness': f'{nowc}/{len(g)} are diagnostic classification; {ind_pct:.0f}% of combinations from Indonesia',
        'imprecision': f'AUC reported in {len(auc)}/{len(g)} combinations; median N {int(g["N (sample)"].median())}',
        'publication_bias': 'not formally assessable; predominance of conference literature indexed in IEEE Xplore',
        'domains_downgraded': '; '.join(rebaixa),
        'levels_downgraded': niveis,
        'certainty': certeza,
    })
grade = pd.DataFrame(linhas_grade)
print(grade[['outcome', 'studies', 'combinations', 'domains_downgraded', 'levels_downgraded', 'certainty']].to_string(index=False))
grava(grade, 'grade_certainty_by_domain.csv')

## 16. Final figures for the manuscript

Copies to `jhpn_revisions/figures/` the set that replaces the figures of the article. Figure 3 (PROBAST+AI, 17 Aug 2026) is already there and is not regenerated here.

In [ ]:
# -- 16.1 copying the final figures --
import shutil

FINAIS = ['fig1_prisma', 'fig2_distribuicao', 'fig4_preditores',
          'figS1_auc_desfecho', 'figS2_desempenho_preditor']
os.makedirs(FIGURAS, exist_ok=True)
for nome in FINAIS:
    for ext in ('png', 'pdf'):
        arq = f'{OUT_DIR}/{nome}_{DATA}.{ext}'
        if os.path.exists(arq):
            shutil.copy2(arq, f'{FIGURAS}/{nome}_{DATA}.{ext}')
            print(f'figures/{nome}_{DATA}.{ext}')
        else:
            print(f'MISSING: {nome}_{DATA}.{ext}')
print('kept unchanged: fig3_probastai_20260817.png/.pdf')

## 17. Tables of the manuscript

Exports the tables in the form in which they enter the article, in English. Table 1, Table 2 and Table 3 go in the body; the others enter as additional material. The formatted `.docx` file is assembled by `scratchpad/docx_mr/build_tabelas.py` from these CSVs.

In [ ]:
# -- 17.1 Table 1: characteristics and performance by combination --
def fmt_num(v, casas=2):
    return 'NR' if pd.isna(v) else f'{v:.{casas}f}'

t1 = df_comb_eleg.copy()
t1['Outcome_EN'] = t1['outcome_family'].map(ROTULO_EN)
t1['Country'] = t1['country'].replace({'USA': 'United States', 'Asia (multinational dataset)': 'Multinational (Asia)'})
t1['Sample'] = t1['N (sample)'].map(lambda v: 'NR' if pd.isna(v) else f'{int(v):,}')
t1['AUC_f'] = t1['AUC'].map(fmt_num)
t1['Accuracy_f'] = t1['Accuracy'].map(fmt_num)
t1['RoB'] = t1['Dev Overall'].astype(str).str.strip().str.capitalize()
finalidade = df_comb.set_index(['ID', 'outcome_family'])['proposed_label']  # one per combination, not per study
t1['Purpose'] = [finalidade.get((i, f), '') for i, f in zip(t1['ID'], t1['outcome_family'])]
t1['Purpose'] = t1['Purpose'].map({'diagnostic classification (nowcasting)': 'Diagnostic',
                                   'prospective prediction': 'Prospective'}).fillna('Undetermined')
tabela1 = t1[['First Author', 'Year', 'Country', 'Sample', 'Age (months)', 'Outcome_EN',
              'Algorithm', 'AUC_f', 'Accuracy_f', 'Purpose', 'RoB']].copy()
tabela1.columns = ['First author', 'Year', 'Country', 'N', 'Age (months)', 'Outcome',
                   'Best model', 'AUC', 'Accuracy', 'Model purpose', 'Risk of bias']
tabela1 = tabela1.sort_values(['Year', 'First author', 'Outcome'])
grava(tabela1, 'table1_study_characteristics.csv')
print(tabela1.head(6).to_string(index=False))

In [ ]:
# -- 17.2 Table 2: performance and heterogeneity by outcome --
t2 = pd.read_csv(f'{OUT_DIR}/table2_nova.csv', sep=';', encoding='utf-8-sig')
het = pd.read_csv(f'{OUT_DIR}/heterogeneity.csv', sep=';', encoding='utf-8-sig')
t2 = t2.merge(het[['outcome_family', 'N_min', 'N_max', 'prev_min', 'prev_max', 'countries']],
              left_on='outcome', right_on='outcome_family', how='left')
tabela2 = pd.DataFrame({
    'Outcome': t2['outcome'].map(ROTULO_EN),
    'Combinations': t2['combinations'],
    'AUC, n': t2['AUC_N'],
    'AUC, median (IQR)': [f'{m:.2f} ({i})' if pd.notna(m) else 'NR'
                          for m, i in zip(t2['AUC_median'], t2['AUC_IQR'])],
    'Accuracy, n': t2['Accuracy_N'],
    'Accuracy, median (IQR)': [f'{m:.2f} ({i})' if pd.notna(m) else 'NR'
                               for m, i in zip(t2['Accuracy_median'], t2['Accuracy_IQR'])],
    'Sample size, range': [f'{int(a):,} to {int(b):,}' if pd.notna(a) else 'NR'
                           for a, b in zip(t2['N_min'], t2['N_max'])],
    'Outcome prevalence, range (%)': [f'{a:.1f} to {b:.1f}' if pd.notna(a) else 'NR'
                                      for a, b in zip(t2['prev_min'], t2['prev_max'])],
    'Countries': t2['countries'],
})
grava(tabela2, 'table2_performance_by_outcome.csv')
print(tabela2.to_string(index=False))

In [ ]:
# -- 17.3 Table 3: certainty of the evidence --
g = pd.read_csv(f'{OUT_DIR}/grade_certainty.csv', sep=';', encoding='utf-8-sig')
CERTEZA_EN = {'very low': 'Very low', 'low': 'Low', 'moderate': 'Moderate', 'high': 'High'}
def serio(campo, marcado):
    return 'Very serious' if 'very serious' in str(campo) else ('Serious' if marcado else 'Not serious')
tabela3 = pd.DataFrame({
    'Outcome': g['outcome'].map(ROTULO_EN),
    'Studies': g['studies'],
    'Risk of bias': ['Very serious' if 'risk of bias' in d else 'Not serious' for d in g['domains_downgraded']],
    'Inconsistency': ['Serious' if 'inconsistency' in d else 'Not serious' for d in g['domains_downgraded']],
    'Indirectness': ['Serious' if 'indirectness' in d else 'Not serious' for d in g['domains_downgraded']],
    'Imprecision': ['Serious' if 'imprecision' in d else 'Not serious' for d in g['domains_downgraded']],
    'Publication bias': 'Not assessed',
    'Certainty': g['certainty'].map(CERTEZA_EN),
})
grava(tabela3, 'table3_grade_certainty.csv')
print(tabela3.to_string(index=False))

In [ ]:
# -- 17.4 tables of the additional material --
# S2: outcomes extracted outside the declared scope (R2.5)
s2 = pd.read_csv(f'{OUT_DIR}/auditoria_desfechos_R2_5.csv', sep=';', encoding='utf-8-sig')
s2 = s2.rename(columns={'ID': 'Study ID', 'First Author': 'First author', 'outcome_family': 'Outcome extracted'})
grava(s2, 'supplement_ineligible_outcomes.csv')

# S3: PROBAST+AI by domain
s3 = pd.read_csv(f'{OUT_DIR}/probastai_domains_intermediate.csv', sep=';', encoding='utf-8-sig')
s3['part'] = s3['part'].map({'development': 'Model development', 'evaluation': 'Model evaluation'})
s3 = s3.rename(columns={'part': 'Part', 'dominio': 'Domain', 'Low_pct': 'Low (%)',
                        'High_pct': 'High (%)', 'Unclear_pct': 'Unclear (%)'})
grava(s3[['Part', 'Domain', 'Low', 'High', 'Unclear', 'Low (%)', 'High (%)', 'Unclear (%)']],
      'supplement_probastai_domains.csv')

# S4: contribution by database
s4 = pd.read_csv(f'{OUT_DIR}/busca_atualizada.csv', sep=';', encoding='utf-8-sig')
s4 = s4.rename(columns={'base': 'Database', 'busca': 'Search', 'registros': 'Records retrieved',
                        'fonte_da_contagem': 'Source of the count',
                        'estudos_incluidos_recuperados': 'Included studies retrieved (of 38)'})
grava(s4, 'supplement_records_by_database.csv')

# S5: verificacao da extracao contra os PDFs
s5 = pd.read_csv(f'{OUT_DIR}/auditoria_v7_por_estudo.csv', sep=';', encoding='utf-8-sig')
s5 = s5[['ID', 'ano', 'n_divergences', 'n_high', 'finalidade_correta',
         'antropometria_do_desfecho_entre_preditores']]
s5.columns = ['Study ID', 'Year', 'Divergences', 'Of which affecting the synthesis',
              'Model purpose after verification', 'Outcome-dimension anthropometry among predictors']
grava(s5.sort_values('Study ID'), 'supplement_data_verification.csv')
print('tables of the additional material written')

## 18. Next step

1. IVS checks `resultados/classificacao_finalidade_TEMPLATE.csv` (columns `rotulo_IVS` and `leakage_IVS`) and `auditoria_desfechos_R2_5.csv`; once the template is marked, rerun sections 10 and 12.
2. IVS decides, in `auditoria_v7_divergencias.csv` (column `aceito_IVS`), which corrections enter a v8 of the extraction. v7 is not altered by this notebook.
3. Decide the wording for the outcomes outside the scope (audit R2.5) and for the combined overweight/obesity category.
4. Take `numeros_manuscrito.csv` to the substitutions in the MARKED Google Doc (together with the new figures in `resultados/`).
5. GRADE (T2.2) and PRISMA/re-search (T1.1, T1.7, R2.4) are left to their own notebooks, after the methodological decisions.